# Stage 2 — Flow Matching to Class Prototypes

This independent Colab notebook adds a Flow Matching (FM) layer on top of the **frozen Stage 1 image-derived class prototypes** (`02_image_prototypes.ipynb`), for ResNet-18 and DINOv2 ViT-S/14 on DTD, FGVC-Aircraft, and Flowers-102. It reuses Stage 1's cached frozen features, official splits, K-shot subsets, and seeds unchanged, so that any accuracy difference is attributable to the FM layer alone. See `ref/stage_2.pdf` for the authoritative spec and `doc/TODO_stage2.md` for the task checklist this notebook implements; `.agents/AGENTS.md` documents the Drive-persistence and cache-and-skip conventions followed throughout.

**Design decisions made explicit here (the PDF leaves them implicit):**
- FM operates on **L2-normalized** frozen features, matching the space Stage 1's image prototypes were already built in (`normalize → mean → normalize`). Interpolating between a raw-scale feature and a unit-norm prototype would make the straight-line path geometrically arbitrary.
- **Standard FM trains one network per (dataset, encoder, K, seed)**, evaluated at both `T=4` and `T=12` at inference (its loss has no `T` dependency). **Rolled-out FM trains two separate networks per setting**, one per `T`, since `T` is baked into its training objective — a `T=4`-trained rollout network is never evaluated at `T=12` or vice versa.
- Checkpoint selection mirrors Stage 1's "highest validation accuracy" rule. For rolled-out training this is unambiguous (accuracy at that network's own `T`). For standard training, checkpoint on the **mean of validation accuracy at every `T` in `T_values`**, since one standard network is later judged at all of them — this avoids arbitrarily privileging one `T` during training that the network is not uniquely optimized for.

Run all cells top to bottom. This notebook never re-runs an encoder and never recomputes a Stage 1 prototype — if a required Stage 1 artifact is missing, it fails loudly rather than silently regenerating it under possibly different conditions.

## 1. Install dependencies
No `torchvision`/`open_clip` needed — this notebook only ever touches cached feature tensors and saved prototypes, never raw images or an encoder.

In [ ]:
%pip -q install scikit-learn pandas seaborn tqdm


## 2. Mount Drive and configure paths
Reuses the same Drive project root as Stages 1's notebooks, so the existing feature caches and prototype artifacts are visible without copying anything.

In [ ]:
from pathlib import Path
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Flow-Matching')
else:
    PROJECT_ROOT = Path('/content/Flow-Matching')
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'flow_matching'
STAGE1_PROTOTYPE_ROOT = PROJECT_ROOT / 'outputs' / 'image_prototypes'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Stage 1 image-prototype outputs:', STAGE1_PROTOTYPE_ROOT)


## 3. Imports, hardware, and the Stage 2 plan
The dataset/encoder/shot/seed plan is loaded from the same `stage1_config.json` Stage 1 wrote, so Stage 2 automatically inherits the exact comprehensive-grid scope decision (all 3 datasets, both encoders on every dataset) without re-declaring it. A `flow_matching` section is added to that same config file (versioned like `linear_probe`/`zero_shot_clip` already are) holding the FM-specific settings the PDF leaves as "no need for extensive search" choices: velocity-network architecture, `T` values, and training hyperparameters.

In [ ]:
import copy, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'none')

DEFAULT_FM_CONFIG = {
    'config_revision': 1,
    'T_values': [4, 12],
    'hidden_dim': 512,
    'num_hidden_layers': 2,
    'batch_size': 64,
    'max_epochs': 200,
    'early_stopping_patience': 25,
    'lr_patience': 10,
    'min_delta': 1e-4,
    'checkpoint_interval': 10,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'resume_training': True,
}
config_paths = [
    PROJECT_ROOT / 'stage1_config.json',
    Path.cwd() / 'stage1_config.json',
    Path('/content/stage1_config.json'),
]
config_path = next((p for p in config_paths if p.exists()), None)
if config_path is None:
    raise FileNotFoundError('stage1_config.json not found. Run 01_linear_probe.ipynb first so the shared experiment plan exists.')
with config_path.open() as f:
    STAGE1 = json.load(f)
print('Loaded configuration from:', config_path)

if STAGE1.get('flow_matching', {}).get('config_revision', 0) < DEFAULT_FM_CONFIG['config_revision']:
    STAGE1['flow_matching'] = copy.deepcopy(DEFAULT_FM_CONFIG)
    config_path.write_text(json.dumps(STAGE1, indent=2))
    print('Added/updated flow_matching config section, revision:', STAGE1['flow_matching']['config_revision'])

PLAN, FM = STAGE1['plan'], STAGE1['flow_matching']
DATASETS = PLAN['datasets']
ENCODERS_BY_DATASET = PLAN['encoders_by_dataset']
ENABLED_PAIRS = [(d, e) for d in DATASETS for e in ENCODERS_BY_DATASET[d]]
SHOT_SETTINGS = PLAN['shots']
SUBSET_SEEDS = PLAN['subset_seeds']
INIT_SEEDS = PLAN['init_seeds']
CACHE_SCHEMA_VERSION = STAGE1['feature_cache']['schema_version']
CACHE_ROOT = PROJECT_ROOT / 'feature_cache' / f'v{CACHE_SCHEMA_VERSION}'
max_encoders = max(len(v) for v in ENCODERS_BY_DATASET.values())

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(0)
print('Datasets:', DATASETS, '| T values:', FM['T_values'])
print('Enabled pairs:', ENABLED_PAIRS)


## 4. Load Stage 1 frozen features and image prototypes
Both functions only ever read from Drive; neither ever re-runs an encoder or recomputes a prototype. A missing artifact raises immediately with a pointer to which Stage 1 notebook produces it, rather than silently falling back to something else.

In [ ]:
def load_cached_features(dataset_name, encoder_name, split):
    cache = CACHE_ROOT / f'{dataset_name}__{encoder_name}__{split}.pt'
    if not cache.exists():
        raise FileNotFoundError(f'Missing Stage 1 feature cache: {cache}. Run 01_linear_probe.ipynb (Section 5) first.')
    return torch.load(cache, map_location='cpu', weights_only=False)

def load_stage1_prototypes(dataset_name, encoder_name, shot, seed):
    run_dir = STAGE1_PROTOTYPE_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / f'seed_{seed}'
    proto_path = run_dir / 'prototypes.pt'
    if not proto_path.exists():
        raise FileNotFoundError(f'Missing Stage 1 image prototype: {proto_path}. Run 02_image_prototypes.ipynb first.')
    artifact = torch.load(proto_path, map_location='cpu', weights_only=False)
    return artifact['prototypes'], artifact['classes']

feature_bank = {}
for dataset_name, encoder_name in ENABLED_PAIRS:
    feature_bank[(dataset_name, encoder_name)] = {
        split: load_cached_features(dataset_name, encoder_name, split) for split in ('train', 'val', 'test')
    }
print('Loaded cached features for:', list(feature_bank))


## 5. Balanced subsets and the Euler integrator
`balanced_indices` is copied verbatim from Stage 1 (`01_linear_probe.ipynb` / `02_image_prototypes.ipynb`) — it must reproduce the *exact* same K-shot subset for a given seed, since Stage 2's training pairs `(z_i, p_{y_i})` need to come from the same images the corresponding Stage 1 prototype was built from. `euler_rollout` is the single integrator shared by inference, rolled-out training, and the trajectory visualization later — it is fully differentiable (plain tensor ops in a Python loop), which is exactly what rolled-out training needs to backpropagate through the whole step sequence.

In [ ]:
def balanced_indices(labels, k, seed):
    labels = np.asarray(labels); rng = np.random.default_rng(seed); chosen = []
    for c in np.unique(labels):
        idx = np.flatnonzero(labels == c)
        if len(idx) < k: raise ValueError(f'class {c} has {len(idx)} samples, fewer than K={k}')
        chosen.extend(rng.choice(idx, size=k, replace=False).tolist())
    return np.asarray(sorted(chosen))

def euler_rollout(v_theta, z0, T, return_trajectory=False):
    z = z0
    trajectory = [z0.detach().clone()] if return_trajectory else None
    for k in range(T):
        t = torch.full((z.shape[0],), k / T, device=z.device, dtype=z.dtype)
        z = z + (1.0 / T) * v_theta(z, t)
        if return_trajectory: trajectory.append(z.detach().clone())
    return (z, trajectory) if return_trajectory else z

def classify_cosine(z, prototypes):
    scores = F.normalize(z, dim=1) @ F.normalize(prototypes, dim=1).T
    return scores.argmax(1), scores


## 6. Velocity network and checkpointed training

`VelocityNet` is the small MLP the PDF suggests: 2 hidden layers, width 512, SiLU activations, scalar `t` concatenated to the feature. `train_velocity_network` trains either objective (`mode='standard'` or `mode='rollout'`) with the same checkpoint/resume machinery `01_linear_probe.ipynb`'s `train_linear_run` uses — checkpoint on best validation accuracy with early stopping, periodic `latest.pt` snapshots so a multi-hour grid survives a Colab disconnect, and a config-compatibility check before resuming so a changed setting is detected and retrained rather than silently continuing from a stale checkpoint.

Training pairs `(z_i, p_{y_i})` are built by indexing the prototype tensor with the training labels (`prototypes[yb]`) — this works directly because Stage 1's `prototypes.pt` stores one row per class index in label order, the same convention `02_image_prototypes.ipynb` classifies against.

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_hidden_layers):
        super().__init__()
        layers = []
        in_dim = feature_dim + 1
        for _ in range(num_hidden_layers):
            layers += [nn.Linear(in_dim, hidden_dim), nn.SiLU()]
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, feature_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z, t):
        if t.dim() == 0: t = t.expand(z.shape[0])
        return self.net(torch.cat([z, t.view(-1, 1).to(z.dtype)], dim=1))

def validation_accuracies(net, x_val, y_val, prototypes, T_list):
    accs = {}
    with torch.no_grad():
        for T in T_list:
            z_hat = euler_rollout(net, x_val, T)
            pred, _ = classify_cosine(z_hat, prototypes)
            accs[T] = (pred == y_val).float().mean().item()
    return accs

def train_velocity_network(mode, x_train, y_train, prototypes, x_val, y_val, run_dir,
                            dataset_name, encoder_name, shot, subset_seed, init_seed, T=None):
    """x_train/y_train/x_val/y_val/prototypes must already be on DEVICE. `mode` is 'standard' or
    'rollout'; for 'rollout', T is required and fixed for the whole run (training and inference use
    the same T, per the PDF). Returns (net, history, best_val_accuracy, runtime_seconds, start_epoch)."""
    seed_everything(init_seed)
    feature_dim = x_train.shape[1]; num_classes = prototypes.shape[0]
    net = VelocityNet(feature_dim, FM['hidden_dim'], FM['num_hidden_layers']).to(DEVICE)
    optimizer = torch.optim.AdamW(net.parameters(), lr=FM['learning_rate'], weight_decay=FM['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=FM['lr_patience'])
    eval_T_list = FM['T_values'] if mode == 'standard' else [T]
    best_acc, best_state, history, start_epoch, stale_epochs = -1.0, None, [], 1, 0

    checkpoint_dir = run_dir / 'checkpoints'; checkpoint_dir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = checkpoint_dir / 'latest.pt', checkpoint_dir / 'best.pt'
    run_config = dict(dataset=dataset_name, encoder=encoder_name, shot=str(shot), subset_seed=subset_seed, init_seed=init_seed,
                       mode=mode, T=T, feature_dim=feature_dim, num_classes=num_classes,
                       hidden_dim=FM['hidden_dim'], num_hidden_layers=FM['num_hidden_layers'],
                       learning_rate=FM['learning_rate'], weight_decay=FM['weight_decay'])
    (run_dir / 'config.json').write_text(json.dumps(run_config, indent=2))

    if FM['resume_training'] and latest_path.exists():
        checkpoint = torch.load(latest_path, map_location=DEVICE, weights_only=False)
        old_config = checkpoint.get('config', {})
        required_keys = ['dataset', 'encoder', 'shot', 'subset_seed', 'init_seed', 'mode', 'T',
                          'feature_dim', 'num_classes', 'hidden_dim', 'num_hidden_layers']
        if all(old_config.get(k) == run_config.get(k) for k in required_keys):
            net.load_state_dict(checkpoint['model_state_dict']); optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if checkpoint.get('scheduler_state_dict') is not None: scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            start_epoch = checkpoint['epoch'] + 1; best_acc = checkpoint['best_val_accuracy']
            best_state = checkpoint['best_model_state_dict']; history = checkpoint['history']
            stale_epochs = checkpoint.get('stale_epochs', 0)
            print(f'Resuming {run_dir} from epoch {start_epoch}')
        else:
            print(f'Checkpoint configuration changed for {run_dir}; starting a fresh run.')

    def save_checkpoint(path, epoch):
        torch.save(dict(epoch=epoch, model_state_dict=net.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                         scheduler_state_dict=scheduler.state_dict(), best_model_state_dict=best_state,
                         best_val_accuracy=best_acc, stale_epochs=stale_epochs, history=history, config=run_config), path)

    started = time.perf_counter()
    n_train = x_train.shape[0]
    for epoch in range(start_epoch, FM['max_epochs'] + 1):
        generator = torch.Generator().manual_seed(init_seed * 100000 + epoch)
        perm = torch.randperm(n_train, generator=generator)
        net.train(); train_total = 0.0
        for start in range(0, n_train, FM['batch_size']):
            batch_idx = perm[start:start + FM['batch_size']]
            zb, yb = x_train[batch_idx], y_train[batch_idx]
            pb = prototypes[yb]
            optimizer.zero_grad(set_to_none=True)
            if mode == 'standard':
                t = torch.rand(zb.shape[0], device=DEVICE)
                zt = (1 - t.view(-1, 1)) * zb + t.view(-1, 1) * pb
                target_velocity = pb - zb
                loss = F.mse_loss(net(zt, t), target_velocity)
            else:
                z_hat_T = euler_rollout(net, zb, T)
                loss = F.mse_loss(z_hat_T, pb)
            loss.backward(); optimizer.step()
            train_total += loss.item() * len(batch_idx)
        val_accs = validation_accuracies(net, x_val, y_val, prototypes, eval_T_list)
        val_acc = float(np.mean(list(val_accs.values())))
        record = dict(epoch=epoch, train_loss=train_total / n_train, val_accuracy=val_acc)
        record.update({f'val_accuracy_T{T_}': a for T_, a in val_accs.items()})
        history.append(record)
        if val_acc > best_acc + FM['min_delta']:
            best_acc, best_state = val_acc, copy.deepcopy(net.state_dict()); stale_epochs = 0; save_checkpoint(best_path, epoch)
        else:
            stale_epochs += 1
        scheduler.step(val_acc)
        if epoch % FM['checkpoint_interval'] == 0: save_checkpoint(latest_path, epoch)
        if stale_epochs >= FM['early_stopping_patience']:
            save_checkpoint(latest_path, epoch); print(f'Early stopping at epoch {epoch}'); break
    if history: save_checkpoint(latest_path, history[-1]['epoch'])
    if best_state is None: best_state = copy.deepcopy(net.state_dict())
    runtime_seconds = time.perf_counter() - started
    net.load_state_dict(best_state); net.eval()
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    return net, history, best_acc, runtime_seconds, start_epoch

def evaluate_split(net, x, y, prototypes, T):
    with torch.no_grad():
        z_hat = euler_rollout(net, x, T)
        pred, _ = classify_cosine(z_hat, prototypes)
        acc = (pred == y).float().mean().item()
    return acc, pred.cpu()

def load_trained_network(dataset_name, encoder_name, shot, subset_seed, init_seed, mode_tag, feature_dim):
    run_tag = f'subset_{subset_seed}__init_{init_seed}'
    run_dir = OUTPUT_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / mode_tag / run_tag
    checkpoint = torch.load(run_dir / 'checkpoints' / 'best.pt', map_location=DEVICE, weights_only=False)
    net = VelocityNet(feature_dim, FM['hidden_dim'], FM['num_hidden_layers']).to(DEVICE)
    state = checkpoint.get('best_model_state_dict') or checkpoint['model_state_dict']
    net.load_state_dict(state); net.eval()
    return net


## 7. Run the complete experiment grid

`FORCE_RETRAIN = False` by default: `load_completed_metrics`/`load_completed_history` check whether a setting's `metrics.json`/`test_predictions.npy`/`history.csv` already exist on Drive before training anything, mirroring the skip-if-already-saved pattern in `01_linear_probe.ipynb`. This matters more here than in Stage 1: the full grid trains **3 networks per (dataset, encoder, K, seed)** — one standard, two rolled-out (`T=4`, `T=12`) — for `6 pairs × 3 shots × 3 repetitions × 3 networks = 162` trainings in total, so re-running the notebook after a disconnect should only ever retrain what is genuinely missing.

Standard FM is trained **once** per setting and evaluated at both `T` values from that single checkpoint; rolled-out FM is trained and evaluated separately per `T`. This is the asymmetry documented in the title cell and in `doc/TODO_stage2.md` — getting it backwards would silently double standard-FM's training cost for no benefit, or (worse) evaluate a rolled-out network at a `T` it was never trained for.

In [ ]:
def load_completed_metrics(eval_dir):
    metrics_path, pred_path = eval_dir / 'metrics.json', eval_dir / 'test_predictions.npy'
    if not (metrics_path.exists() and pred_path.exists()):
        return None
    return json.loads(metrics_path.read_text())

def load_completed_history(run_dir):
    history_path = run_dir / 'history.csv'
    return pd.read_csv(history_path).to_dict('records') if history_path.exists() else None

FORCE_RETRAIN = False  # set True to ignore saved FM results on Drive and retrain everything

def run_fm_setting(dataset_name, encoder_name, shot, subset_seed, init_seed):
    bank = feature_bank[(dataset_name, encoder_name)]
    x_train_raw, y_train_full = bank['train']['features'].float(), bank['train']['labels'].long()
    if shot != 'full':
        idx = torch.as_tensor(balanced_indices(y_train_full.numpy(), int(shot), subset_seed))
        x_train_raw, y_train = x_train_raw[idx], y_train_full[idx]
    else:
        y_train = y_train_full
    x_train = F.normalize(x_train_raw, dim=1).to(DEVICE); y_train = y_train.to(DEVICE)
    x_val = F.normalize(bank['val']['features'].float(), dim=1).to(DEVICE); y_val = bank['val']['labels'].long().to(DEVICE)
    x_test = F.normalize(bank['test']['features'].float(), dim=1).to(DEVICE); y_test = bank['test']['labels'].long().to(DEVICE)
    proto_seed = subset_seed if shot != 'full' else 0
    prototypes, _classes = load_stage1_prototypes(dataset_name, encoder_name, shot, proto_seed)
    prototypes = prototypes.to(DEVICE)
    run_tag = f'subset_{subset_seed}__init_{init_seed}'

    results = []

    # --- standard FM: one network, evaluated at every T ---
    standard_dir = OUTPUT_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / 'standard' / run_tag
    standard_eval_dirs = {T: standard_dir / f'eval_T{T}' for T in FM['T_values']}
    cached_metrics = {T: load_completed_metrics(standard_eval_dirs[T]) for T in FM['T_values']}
    if not FORCE_RETRAIN and all(m is not None for m in cached_metrics.values()):
        print(f'Loaded saved FM result: {dataset_name} | {encoder_name} | {shot} | standard | subset {subset_seed} | init {init_seed}')
        results.extend(cached_metrics.values())
        standard_history = load_completed_history(standard_dir)
    else:
        print(f'\n{dataset_name} | {encoder_name} | {shot} | standard | subset {subset_seed} | init {init_seed}')
        net, standard_history, best_val_acc, runtime_seconds, resumed_from = train_velocity_network(
            'standard', x_train, y_train, prototypes, x_val, y_val, standard_dir,
            dataset_name, encoder_name, shot, subset_seed, init_seed)
        for T in FM['T_values']:
            test_acc, test_pred = evaluate_split(net, x_test, y_test, prototypes, T)
            standard_eval_dirs[T].mkdir(parents=True, exist_ok=True)
            np.save(standard_eval_dirs[T] / 'test_predictions.npy', test_pred.numpy())
            metrics = dict(dataset=dataset_name, encoder=encoder_name, shot=str(shot), subset_seed=subset_seed, init_seed=init_seed,
                            variant='standard', T=T, best_val_accuracy=best_val_acc, test_accuracy=test_acc,
                            runtime_seconds=runtime_seconds, resumed_from_epoch=resumed_from)
            (standard_eval_dirs[T] / 'metrics.json').write_text(json.dumps(metrics, indent=2))
            results.append(metrics)

    # --- rolled-out FM: separate network + evaluation per T ---
    rollout_histories = {}
    for T in FM['T_values']:
        rollout_dir = OUTPUT_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / f'rollout_T{T}' / run_tag
        eval_dir = rollout_dir / 'eval'
        cached = None if FORCE_RETRAIN else load_completed_metrics(eval_dir)
        if cached is not None:
            print(f'Loaded saved FM result: {dataset_name} | {encoder_name} | {shot} | rollout_T{T} | subset {subset_seed} | init {init_seed}')
            results.append(cached); rollout_histories[T] = load_completed_history(rollout_dir); continue
        print(f'\n{dataset_name} | {encoder_name} | {shot} | rollout_T{T} | subset {subset_seed} | init {init_seed}')
        net, history, best_val_acc, runtime_seconds, resumed_from = train_velocity_network(
            'rollout', x_train, y_train, prototypes, x_val, y_val, rollout_dir,
            dataset_name, encoder_name, shot, subset_seed, init_seed, T=T)
        rollout_histories[T] = history
        test_acc, test_pred = evaluate_split(net, x_test, y_test, prototypes, T)
        eval_dir.mkdir(parents=True, exist_ok=True)
        np.save(eval_dir / 'test_predictions.npy', test_pred.numpy())
        metrics = dict(dataset=dataset_name, encoder=encoder_name, shot=str(shot), subset_seed=subset_seed, init_seed=init_seed,
                        variant='rollout', T=T, best_val_accuracy=best_val_acc, test_accuracy=test_acc,
                        runtime_seconds=runtime_seconds, resumed_from_epoch=resumed_from)
        (eval_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
        results.append(metrics)

    return results, standard_history, rollout_histories

all_fm_results, fm_histories = [], {}
for dataset_name, encoder_name in ENABLED_PAIRS:
    for shot in SHOT_SETTINGS:
        repetitions = [(s, 0) for s in SUBSET_SEEDS] if shot != 'full' else [(None, s) for s in INIT_SEEDS]
        for subset_seed, init_seed in repetitions:
            run_key = subset_seed if shot != 'full' else init_seed
            results, standard_history, rollout_histories = run_fm_setting(dataset_name, encoder_name, shot, subset_seed, init_seed)
            all_fm_results.extend(results)
            fm_histories[(dataset_name, encoder_name, str(shot), run_key)] = dict(
                standard=standard_history, rollout_T4=rollout_histories[4], rollout_T12=rollout_histories[12])
results_df = pd.DataFrame(all_fm_results)
results_df.to_csv(OUTPUT_ROOT / 'run_metrics.csv', index=False)
display(results_df)


## 8. Aggregate accuracy, ΔAcc against the Stage 1 baseline, and the accuracy-vs-K plot

The Stage 1 baseline numbers come straight from `02_image_prototypes.ipynb`'s own `accuracy_summary.csv` — never recomputed here — so `ΔAcc = Acc_FM − Acc_baseline` is a comparison against exactly the numbers already reported for Stage 1, joined on `(dataset, encoder, shot)`.

In [ ]:
fm_summary = results_df.groupby(['dataset', 'encoder', 'shot', 'variant', 'T'], as_index=False).agg(
    mean_accuracy=('test_accuracy', 'mean'), std_accuracy=('test_accuracy', 'std'), runs=('test_accuracy', 'size'))

baseline_summary = pd.read_csv(STAGE1_PROTOTYPE_ROOT / 'accuracy_summary.csv')
baseline_summary = baseline_summary.rename(columns={'mean_accuracy': 'baseline_mean_accuracy', 'std_accuracy': 'baseline_std_accuracy'})
baseline_summary['shot'] = baseline_summary['shot'].astype(str)

comparison = fm_summary.merge(
    baseline_summary[['dataset', 'encoder', 'shot', 'baseline_mean_accuracy', 'baseline_std_accuracy']],
    on=['dataset', 'encoder', 'shot'], how='left')
comparison['delta_acc'] = comparison['mean_accuracy'] - comparison['baseline_mean_accuracy']
comparison.to_csv(OUTPUT_ROOT / 'accuracy_summary_with_baseline.csv', index=False)
display(comparison.style.format({'mean_accuracy': '{:.4f}', 'std_accuracy': '{:.4f}',
                                  'baseline_mean_accuracy': '{:.4f}', 'baseline_std_accuracy': '{:.4f}',
                                  'delta_acc': '{:+.4f}'}))

shot_x = {'5': 0, '10': 1, 'full': 2}
fig, axes = plt.subplots(len(DATASETS), max_encoders, figsize=(7 * max_encoders, 5 * len(DATASETS)), squeeze=False)
for i, dataset_name in enumerate(DATASETS):
    for j, encoder_name in enumerate(ENCODERS_BY_DATASET[dataset_name]):
        ax = axes[i, j]
        base = baseline_summary[(baseline_summary.dataset == dataset_name) & (baseline_summary.encoder == encoder_name)].copy()
        base['x'] = base.shot.map(shot_x); base = base.sort_values('x')
        ax.errorbar(base.x, base.baseline_mean_accuracy, yerr=base.baseline_std_accuracy.fillna(0),
                     marker='s', capsize=4, label='Stage 1 baseline', color='black', linestyle='--')
        part = fm_summary[(fm_summary.dataset == dataset_name) & (fm_summary.encoder == encoder_name)].copy()
        part['x'] = part.shot.map(shot_x)
        for variant, T in [('standard', 4), ('standard', 12), ('rollout', 4), ('rollout', 12)]:
            row = part[(part.variant == variant) & (part.T == T)].sort_values('x')
            ax.errorbar(row.x, row.mean_accuracy, yerr=row.std_accuracy.fillna(0), marker='o', capsize=4, label=f'{variant} T={T}')
        ax.set(title=f'{dataset_name} / {encoder_name}', xticks=[0, 1, 2], xticklabels=['5', '10', 'full'],
               xlabel='training images per class', ylabel='top-1 test accuracy')
        ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'accuracy_vs_training_size.png', dpi=180, bbox_inches='tight'); plt.show()


## 9. Representative training curves

Seed 0 of the 10-shot setting is used as the representative run, matching Stage 1's convention. Standard-FM loss (velocity MSE at a random `t`) and rolled-out loss (final-point MSE after a full `T`-step rollout) are different quantities on different scales — plotted together on a log axis purely to check what the PDF asks for: that both approaches train stably and reach a reasonable solution, not to compare their absolute magnitudes.

In [ ]:
fig, axes = plt.subplots(len(DATASETS), max_encoders, figsize=(7 * max_encoders, 5 * len(DATASETS)), squeeze=False)
for i, dataset_name in enumerate(DATASETS):
    for j, encoder_name in enumerate(ENCODERS_BY_DATASET[dataset_name]):
        ax = axes[i, j]
        hist_bundle = fm_histories[(dataset_name, encoder_name, '10', 0)]
        for mode_tag, label in [('standard', 'standard FM'), ('rollout_T4', 'rollout T=4'), ('rollout_T12', 'rollout T=12')]:
            h = pd.DataFrame(hist_bundle[mode_tag])
            ax.plot(h.epoch, h.train_loss, label=label)
        ax.set(title=f'{dataset_name} / {encoder_name}', xlabel='epoch', ylabel='training loss (MSE)', yscale='log')
        ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'training_curves.png', dpi=180, bbox_inches='tight'); plt.show()


## 10. Feature-space visualization: original vs. standard-FM vs. rolled-out-FM

Per the PDF, the projection is fit **jointly across all three views plus the prototypes** (one PCA fit over the concatenation of original/standard-transported/rollout-transported features and the selected class prototypes), so the three panels share one coordinate frame and a point's movement between panels is directly comparable — not three independently-fit projections that happen to look similar. Same 10 classes, same up-to-30 test examples per class, and same class colors as Stage 1's feature plots, using the representative 10-shot/seed-0 networks (`T=12`).

In [ ]:
fig, axes = plt.subplots(len(DATASETS), max_encoders * 3, figsize=(5 * max_encoders * 3, 4 * len(DATASETS)), squeeze=False)
for i, dataset_name in enumerate(DATASETS):
    selected_classes = np.arange(10)
    base_labels = feature_bank[(dataset_name, 'resnet18')]['test']['labels'].numpy()
    selected_idx = np.concatenate([np.flatnonzero(base_labels == c)[:30] for c in selected_classes])
    for j, encoder_name in enumerate(ENCODERS_BY_DATASET[dataset_name]):
        bank = feature_bank[(dataset_name, encoder_name)]['test']
        z = F.normalize(bank['features'][selected_idx].float(), dim=1).to(DEVICE)
        y = bank['labels'][selected_idx].numpy()
        prototypes, _ = load_stage1_prototypes(dataset_name, encoder_name, '10', 0)
        prototypes = prototypes.to(DEVICE)
        feature_dim = z.shape[1]
        standard_net = load_trained_network(dataset_name, encoder_name, '10', 0, 0, 'standard', feature_dim)
        rollout_net = load_trained_network(dataset_name, encoder_name, '10', 0, 0, 'rollout_T12', feature_dim)
        with torch.no_grad():
            z_standard = euler_rollout(standard_net, z, 12)
            z_rollout = euler_rollout(rollout_net, z, 12)
        joint = torch.cat([z, z_standard, z_rollout, prototypes[selected_classes]]).cpu().numpy()
        xy = PCA(n_components=2).fit_transform(joint)
        n = len(z)
        views = {'original': xy[:n], 'standard FM (T=12)': xy[n:2 * n], 'rollout FM (T=12)': xy[2 * n:3 * n]}
        proto_xy = xy[3 * n:]
        palette = sns.color_palette('tab10', 10)
        for k, (title, pts) in enumerate(views.items()):
            ax = axes[i, 3 * j + k]
            sns.scatterplot(x=pts[:, 0], y=pts[:, 1], hue=y, palette=palette, s=20, alpha=.65, ax=ax, legend=False)
            for c_idx in range(len(selected_classes)):
                ax.scatter(proto_xy[c_idx, 0], proto_xy[c_idx, 1], color=palette[c_idx], marker='X', s=150, edgecolors='black')
            ax.set(title=f'{dataset_name}/{encoder_name}\n{title}', xlabel='PC1', ylabel='PC2')
    for j in range(len(ENCODERS_BY_DATASET[dataset_name]) * 3, max_encoders * 3):
        axes[i, j].axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'feature_space_comparison.png', dpi=180, bbox_inches='tight'); plt.show()


## 11. Flow trajectories

For 3 representative test examples per dataset (one per class, first 3 classes, strongest encoder per dataset), the intermediate Euler states `z_hat_0 ... z_hat_12` from the standard-FM network are plotted together with the original feature, final transported feature, and target prototype, in one PCA fit jointly over that single trajectory plus its prototype (PCA rather than t-SNE, per the PDF, since the projected path needs to stay geometrically interpretable as a trajectory).

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS) * 3, figsize=(4 * len(DATASETS) * 3, 4))
col = 0
for dataset_name in DATASETS:
    encoder_name = ENCODERS_BY_DATASET[dataset_name][-1]
    bank = feature_bank[(dataset_name, encoder_name)]['test']
    y = bank['labels'].numpy()
    chosen_classes = np.arange(3)
    chosen_idx = [int(np.flatnonzero(y == c)[0]) for c in chosen_classes]
    prototypes, _ = load_stage1_prototypes(dataset_name, encoder_name, '10', 0)
    prototypes = prototypes.to(DEVICE)
    feature_dim = prototypes.shape[1]
    net = load_trained_network(dataset_name, encoder_name, '10', 0, 0, 'standard', feature_dim)
    for idx, c in zip(chosen_idx, chosen_classes):
        z0 = F.normalize(bank['features'][idx:idx + 1].float(), dim=1).to(DEVICE)
        with torch.no_grad():
            _, trajectory = euler_rollout(net, z0, 12, return_trajectory=True)
        traj = torch.cat(trajectory).cpu().numpy()
        proto = prototypes[c:c + 1].cpu().numpy()
        xy = PCA(n_components=2).fit_transform(np.concatenate([traj, proto]))
        ax = axes[col]; col += 1
        ax.plot(xy[:-1, 0], xy[:-1, 1], '-o', color='tab:blue', markersize=4, label='trajectory')
        ax.scatter(xy[0, 0], xy[0, 1], color='green', s=120, zorder=5, label='start (z)')
        ax.scatter(xy[-2, 0], xy[-2, 1], color='purple', s=120, marker='s', zorder=5, label='end (z_T)')
        ax.scatter(xy[-1, 0], xy[-1, 1], color='black', s=160, marker='X', zorder=5, label='prototype')
        ax.set(title=f'{dataset_name}/{encoder_name}\nclass {c}', xlabel='PC1', ylabel='PC2')
        ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'flow_trajectories.png', dpi=180, bbox_inches='tight'); plt.show()


## 12. Optional: the flow in reverse

The PDF's optional extension: start from each class prototype (`t=1`) and integrate the *same* learned velocity field backward toward `t=0`, using the standard-FM network. `euler_rollout_reverse` evaluates the field at decreasing time and subtracts rather than adds, the standard way to traverse a learned ODE backward. Plotted jointly with the original test features to see whether the reverse-transported prototypes land back near their class's real feature cloud - a purely qualitative check on how well-behaved (invertible-looking) the learned flow is, not a required result.

In [ ]:
def euler_rollout_reverse(v_theta, z0, T):
    z = z0
    for k in range(T):
        t = torch.full((z.shape[0],), 1.0 - k / T, device=z.device, dtype=z.dtype)
        z = z - (1.0 / T) * v_theta(z, t)
    return z

fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 5), squeeze=False); axes = axes.ravel()
for ax, dataset_name in zip(axes, DATASETS):
    encoder_name = ENCODERS_BY_DATASET[dataset_name][-1]
    selected_classes = np.arange(10)
    base_labels = feature_bank[(dataset_name, 'resnet18')]['test']['labels'].numpy()
    selected_idx = np.concatenate([np.flatnonzero(base_labels == c)[:30] for c in selected_classes])
    bank = feature_bank[(dataset_name, encoder_name)]['test']
    z = F.normalize(bank['features'][selected_idx].float(), dim=1).to(DEVICE)
    y = bank['labels'][selected_idx].numpy()
    prototypes, _ = load_stage1_prototypes(dataset_name, encoder_name, '10', 0)
    prototypes = prototypes.to(DEVICE)
    net = load_trained_network(dataset_name, encoder_name, '10', 0, 0, 'standard', z.shape[1])
    with torch.no_grad():
        reversed_protos = euler_rollout_reverse(net, prototypes[selected_classes], 12)
    joint = torch.cat([z, reversed_protos]).cpu().numpy(); n = len(z)
    xy = PCA(n_components=2).fit_transform(joint)
    palette = sns.color_palette('tab10', 10)
    sns.scatterplot(x=xy[:n, 0], y=xy[:n, 1], hue=y, palette=palette, s=20, alpha=.5, ax=ax, legend=False)
    for c_idx in range(len(selected_classes)):
        ax.scatter(xy[n + c_idx, 0], xy[n + c_idx, 1], color=palette[c_idx], marker='P', s=150, edgecolors='black')
    ax.set(title=f'{dataset_name} / {encoder_name}: reverse-flowed prototypes (P) vs. test features', xlabel='PC1', ylabel='PC2')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'reverse_flow.png', dpi=180, bbox_inches='tight'); plt.show()
print('Artifacts saved to:', OUTPUT_ROOT)
